# Pass / Fail 的 Likert 评分分类表

读取原始 Excel 中的清晰度、流畅度评分，分别按 1–5 级统计 Pass、Fail 及合计数量。百分比的分母为各类别的视频总数；两项评分分别统计，不合并成一个分数。保留所有等级，包括数量为 0 的等级。

这些是基于连续帧检查的初评，不能作为独立人工评分或实时播放流畅度评估的替代。本 Notebook 只做描述统计，不训练模型，也不修改源 Excel。

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# 在项目根目录或 code 目录运行均可；移动项目后可手动修改 PROJECT_ROOT。
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data/labeled_data/0_PASS/0_PASS.xlsx').exists()),
                    Path.home() / 'Desktop/oae')
SOURCES = {
    'Pass': PROJECT_ROOT / 'data/labeled_data/0_PASS/0_PASS.xlsx',
    'Fail': PROJECT_ROOT / 'data/labeled_data/1_FAIL/1_FAIL.xlsx',
}
SCORE_COLUMNS = {'清晰度': '清晰度评分（1–5）', '流畅度': '流畅度评分（1–5）'}
LEVELS = [1, 2, 3, 4, 5]
# 等级名称只是分值的文字展示，不对原评分重新标注。
LEVEL_NAMES = {1: '很差', 2: '较差', 3: '一般', 4: '较好', 5: '很好'}
frames = []
for label, path in SOURCES.items():
    source = pd.read_excel(path)
    required = ['ID', *SCORE_COLUMNS.values()]
    missing = set(required) - set(source.columns)
    if missing:
        raise ValueError(f'{path.name} 缺少列：{sorted(missing)}')
    part = source[required].copy()
    if part['ID'].isna().any() or part['ID'].duplicated().any():
        raise ValueError(f'{path.name} 存在空 ID 或重复 ID')
    for column in SCORE_COLUMNS.values():
        values = pd.to_numeric(part[column], errors='coerce')
        invalid = ~values.isin(LEVELS)
        if invalid.any():
            raise ValueError(f'{path.name} 的 {column} 存在缺失或非法评分：'
                             f'{part.loc[invalid, "ID"].tolist()}')
        part[column] = values.astype(int)
    part.insert(1, '类别', label)
    frames.append(part)
data = pd.concat(frames, ignore_index=True)
if data['ID'].duplicated().any():
    raise ValueError('Pass 与 Fail 之间存在重复 ID，请先核对')
sample_counts = data.groupby('类别').size().reindex(['Pass', 'Fail']).rename('视频数量').to_frame()
sample_counts.loc['合计'] = len(data)
display(sample_counts)


,视频数量
类别,
Pass,304
Fail,47
合计,351


## 按等级统计数量与占比

每张表的 Pass、Fail 占比各自加总为 100%；合计占比以全部 Pass/Fail 视频为分母。

In [2]:
def make_likert_table(column):
    counts = pd.crosstab(data[column], data['类别']).reindex(
        index=LEVELS, columns=['Pass', 'Fail'], fill_value=0)
    table = pd.DataFrame({'等级名称': [LEVEL_NAMES[level] for level in LEVELS]},
                         index=pd.Index(LEVELS, name='Likert等级'))
    for label in ['Pass', 'Fail']:
        table[f'{label}数量'] = counts[label]
        table[f'{label}占比'] = counts[label] / (data['类别'] == label).sum()
    table['合计数量'] = counts.sum(axis=1)
    table['合计占比'] = table['合计数量'] / len(data)
    assert table['合计数量'].sum() == len(data)
    total = {'等级名称': '全部等级'}
    for label in ['Pass', 'Fail']:
        total[f'{label}数量'] = int(counts[label].sum())
        total[f'{label}占比'] = 1.0
    total.update({'合计数量': len(data), '合计占比': 1.0})
    table.loc['合计'] = total
    return table

likert_tables = {metric: make_likert_table(column) for metric, column in SCORE_COLUMNS.items()}
for metric, table in likert_tables.items():
    print(f'{metric}评分分布')
    display(table.style.format({column: '{:.1%}' for column in table.columns if '占比' in column}))


清晰度评分分布
流畅度评分分布


,等级名称,Pass数量,Pass占比,Fail数量,Fail占比,合计数量,合计占比
Likert等级,,,,,,,
1,很差,0,0.0%,0,0.0%,0,0.0%
2,较差,0,0.0%,1,2.1%,1,0.3%
3,一般,76,25.0%,10,21.3%,86,24.5%
4,较好,119,39.1%,19,40.4%,138,39.3%
5,很好,109,35.9%,17,36.2%,126,35.9%
合计,全部等级,304,100.0%,47,100.0%,351,100.0%


,等级名称,Pass数量,Pass占比,Fail数量,Fail占比,合计数量,合计占比
Likert等级,,,,,,,
1,很差,0,0.0%,0,0.0%,0,0.0%
2,较差,0,0.0%,8,17.0%,8,2.3%
3,一般,32,10.5%,25,53.2%,57,16.2%
4,较好,154,50.7%,11,23.4%,165,47.0%
5,很好,118,38.8%,3,6.4%,121,34.5%
合计,全部等级,304,100.0%,47,100.0%,351,100.0%


## 清晰度 × 流畅度交叉表

行是清晰度等级，列是流畅度等级，单元格是视频数量。

In [3]:
joint_tables = {}
for label in ['Pass', 'Fail', '合计']:
    subset = data if label == '合计' else data.loc[data['类别'] == label]
    table = pd.crosstab(subset[SCORE_COLUMNS['清晰度']], subset[SCORE_COLUMNS['流畅度']]).reindex(
        index=LEVELS, columns=LEVELS, fill_value=0)
    table.index.name = '清晰度等级'
    table.columns.name = '流畅度等级'
    assert table.to_numpy().sum() == len(subset)
    table['合计'] = table.sum(axis=1)
    table.loc['合计'] = table.sum(axis=0)
    joint_tables[label] = table
    print(label)
    display(table)


Pass
Fail
合计


流畅度等级,1,2,3,4,5,合计
清晰度等级,,,,,,
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,8,66,2,76
4,0,0,18,69,32,119
5,0,0,6,19,84,109
合计,0,0,32,154,118,304


流畅度等级,1,2,3,4,5,合计
清晰度等级,,,,,,
1,0,0,0,0,0,0
2,0,1,0,0,0,1
3,0,4,5,1,0,10
4,0,3,13,3,0,19
5,0,0,7,7,3,17
合计,0,8,25,11,3,47


流畅度等级,1,2,3,4,5,合计
清晰度等级,,,,,,
1,0,0,0,0,0,0
2,0,1,0,0,0,1
3,0,4,13,67,2,86
4,0,3,31,72,32,138
5,0,0,13,26,87,126
合计,0,8,57,165,121,351


## 各等级对应的视频 ID

用于按等级查找视频；只显示现有评分，不新增评分理由。

In [4]:
groups = []
for metric, column in SCORE_COLUMNS.items():
    for label in ['Pass', 'Fail']:
        for level in LEVELS:
            ids = sorted(data.loc[(data['类别'] == label) & (data[column] == level), 'ID'].tolist())
            groups.append({'评分维度': metric, '类别': label, 'Likert等级': level,
                           '视频数量': len(ids), '视频ID': ', '.join(ids)})
grade_members = pd.DataFrame(groups)
with pd.option_context('display.max_colwidth', None, 'display.max_rows', 25):
    display(grade_members)


,评分维度,类别,Likert等级,视频数量,视频ID
0,清晰度,Pass,1,0,
1,清晰度,Pass,2,0,
2,清晰度,Pass,3,76,"C0034, C0038, C0040, C0052, C0054, C0066, C0068, C0080, C0082, C0094, C0096, C0098, C0108, C0110, C0112, C0122, C0136, C0150, C0152, C0164, C0166, C0178, C0180, C0192, C0194, C0206, C0208, C0220, C0222, C0234, C0236, C0248, C0262, C0264, C0276, C0278, C0290, C0304, C0318, C0320, C0332, C0342, C0346, C0348, C0356, C0360, C0362, C0374, C0376, C0384, C0388, C0390, C0402, C0404, C0416, C0418, C0430, C0444, C0454, C0458, C0468, C0472, C0482, C0486, C0496, C0500, C0514, C0528, C0542, C0556, C0558, C0570, C0584, C0598, C0608, C0612"
3,清晰度,Pass,4,119,"C0006, C0007, C0012, C0019, C0026, C0048, C0062, C0076, C0090, C0097, C0099, C0104, C0111, C0118, C0124, C0126, C0127, C0132, C0138, C0140, C0146, C0160, C0161, C0162, C0174, C0188, C0189, C0202, C0216, C0230, C0244, C0250, C0255, C0258, C0259, C0261, C0265, C0275, C0279, C0286, C0292, C0300, C0306, C0314, C0328, C0334, C0367, C0377, C0381, C0391, C0395, C0398, C0399, C0401, C0405, C0412, C0413, C0415, C0419, C0426, C0432, C0439, C0440, C0446, C0451, C0457, C0460, C0461, C0465, C0467, C0469, C0471, C0474, C0475, C0478, C0479, C0485, C0488, C0489, C0502, C0510, C0516, C0523, C0524, C0525, C0530, C0535, C0537, C0538, C0539, C0541, C0544, C0545, C0549, C0552, C0553, C0555, C0559, C0563, C0566, C0567, C0572, C0573, C0577, C0580, C0581, C0583, C0586, C0591, C0594, C0595, C0597, C0600, C0601, C0605, C0609, C0611, C0614, C0615"
4,清晰度,Pass,5,109,"C0002, C0003, C0013, C0021, C0027, C0033, C0035, C0037, C0041, C0044, C0049, C0055, C0059, C0063, C0064, C0065, C0069, C0073, C0077, C0078, C0079, C0083, C0086, C0089, C0093, C0095, C0100, C0101, C0107, C0109, C0113, C0114, C0115, C0117, C0119, C0120, C0121, C0123, C0125, C0128, C0129, C0131, C0133, C0135, C0137, C0139, C0143, C0147, C0149, C0157, C0163, C0171, C0175, C0177, C0185, C0191, C0195, C0199, C0203, C0205, C0209, C0213, C0215, C0217, C0219, C0223, C0227, C0231, C0237, C0247, C0251, C0283, C0285, C0287, C0289, C0291, C0293, C0297, C0303, C0307, C0315, C0317, C0321, C0325, C0329, C0331, C0335, C0339, C0345, C0353, C0359, C0363, C0423, C0427, C0429, C0433, C0437, C0441, C0443, C0447, C0453, C0495, C0499, C0501, C0503, C0513, C0517, C0527, C0531"
5,清晰度,Fail,1,0,
6,清晰度,Fail,2,1,C0260
7,清晰度,Fail,3,10,"C0005, C0010, C0020, C0022, C0024, C0091, C0092, C0272, C0385, C0386"
8,清晰度,Fail,4,19,"C0017, C0047, C0085, C0087, C0103, C0105, C0106, C0134, C0190, C0269, C0273, C0274, C0301, C0313, C0327, C0369, C0383, C0455, C0483"
9,清晰度,Fail,5,17,"C0016, C0031, C0045, C0061, C0181, C0241, C0311, C0324, C0338, C0347, C0349, C0355, C0361, C0493, C0497, C0507, C0521"
